Проверить доступность API

https://vllm.ru.tuna.am/docs#

# **Суммаризатор**

In [32]:
# @title Конфигич
file_path = "/content/text1_source.txt"
output_path = "/content/medical_summary.json"
temperature = 0.2
max_tokens = 2000
enable_thinking = False

В эталоне промпт, парсинг json и версионное сохранение файлов `[имя файла МК]_summary_v[n]`

дополнительно извлечение logprobs для токен-уровневых метрик.

In [33]:
# @title Промпт эталонной суммаризации + logprobs

import openai
import json
import re
import math
from pathlib import Path
from typing import Dict, List, Optional
from dataclasses import dataclass

# ============================================
# 1. НАСТРОЙКА КЛИЕНТА
# ============================================

client = openai.OpenAI(
    base_url="https://vllm.ru.tuna.am/v1",
    api_key=""  # Ваш ключ
)

model = "Qwen/Qwen3.5-9B"

# ============================================
# 2. ЧТЕНИЕ ФАЙЛА
# ============================================

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        file_content = f.read()
except FileNotFoundError:
    print(f"⚠️ Ошибка: Файл '{file_path}' не найден.")
    file_content = ""

# ============================================
# 3. ПРОМПТ (ваш существующий)
# ============================================

system_prompt = '''
Ты — помощник врача-рентгенолога для подготовки к КТ органов брюшной полости (ОБП).
Твоя задача — составить структурированное медицинское резюме на основе предоставленной информации о пациенте из ЭМК.

### ПРАВИЛА:
Возвращай ТОЛЬКО чистый JSON (без ``` и любых маркеров).
Все даты — в формате YYYY-MM-DD.
Точка отсчёта: дата последнего документа в карте. Фильтруй данные за 1 год от неё.
Если дат нет — включай все данные.
Хронические заболевания и аллергии — ВСЕГДА включай, независимо от давности.
Если данных нет — пиши "Не указано". Если конфликт — бери последнее по дате.
Не выдумывай. Цитируй источник: "диагноз (дата: YYYY-MM-DD)".
В секции "на_что_обратить_внимание" — только патологии ОБП, влияющие на интерпретацию КТ.

### JSON-СТРУКТУРА:
{
  "пациент": "ФИО, возраст, рост, вес(последняя запись)",
  "жалобы": "Только жалобы, ставшие причиной выполнения КТ",
  "анамнез_заболевания": "Дата начала: YYYY-MM-DD, динамика, диагнозы, лечение",
  "анамнез_жизни": {
    "хронические_заболевания": ["Болезнь (начало: YYYY-MM-DD)"],
    "история болезни": ["Болезнь (год: YYYY-MM-DD)"],
    "вредные_привычки": ["Описание"],
    "операции_травмы": ["Описание (дата: YYYY-MM-DD)"],
    "аллергии": ["Аллерген"]
  },
  "лабораторные_данные": [
    {"показатель": "...", "значение": "...", "дата": "YYYY-MM-DD"}
  ],
  "инструментальные_данные": [
    {"исследование": "...", "дата": "YYYY-MM-DD", "находки": "..."}
  ],
  "цель_исследования": "Краткий вывод, что нужно подтвердить или исключить",
  "на_что_обратить_внимание_для_задачи_КТ_ОБП": ["пункт 1", "пункт 2"]
}
'''

user_prompt = "Суммаризируй медицинский текст:"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": [{"type": "text", "text": f"{user_prompt}\n\n{file_content}"}]}
]

# ============================================
# 4. ЗАПРОС С LOGPROBS 🔑
# ============================================

response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=temperature,
    max_tokens=max_tokens,
    logprobs=True,           # 🔑 ВКЛЮЧАЕМ logprobs
    top_logprobs=10,         # 🔑 Топ-10 кандидатов для энтропии
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": False}
    }
)

# ============================================
# 5. ИЗВЛЕЧЕНИЕ ДАННЫХ
# ============================================

raw = response.choices[0].message.content.strip()
logprobs_data = response.choices[0].logprobs.content  # 🔑 Список токенов с logprobs

def extract_json_from_response(text):
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1:
        return text[start:end+1]
    return text

clean = extract_json_from_response(raw)

try:
    result = json.loads(clean)
except json.JSONDecodeError as e:
    print(f"⚠️ Ошибка парсинга: {e}")
    with open('debug_raw_response.txt', 'w', encoding='utf-8') as f:
        f.write(raw)
    result = {"error": "JSON parse failed", "raw": raw}

# ============================================
# 6. ВЕРСИОНИРОВАНИЕ (ваш код)
# ============================================

input_file_base_name = Path(file_path).stem
output_path_obj = Path(output_path)
parent_dir = output_path_obj.parent
base_name = f"{input_file_base_name}_summary"
suffix = output_path_obj.suffix

version = 0
existing_files = list(parent_dir.glob(f"{base_name}_v*{suffix}"))

for f in existing_files:
    match = re.search(r'_v(\d+)', f.stem)
    if match:
        current_version = int(match.group(1))
        if current_version > version:
            version = current_version

version += 1
new_output_filename = f"{base_name}_v{version}{suffix}"
new_output_path = parent_dir / new_output_filename

# Сохраняем результат (метрики добавим в ячейке 16)
with open(new_output_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"✅ Результат сохранён в {new_output_path}")
print(f"📊 Токенов с logprobs: {len(logprobs_data)}")

# ============================================
# 7. ПЕРЕДАЧА ДАННЫХ В ЯЧЕЙКУ 16
# ============================================

# Эти переменные будут доступны в следующей ячейке
summary_text = raw
summary_logprobs = logprobs_data
summary_result = result
summary_path = new_output_path

✅ Результат сохранён в /content/text1_source_summary_v6.json
📊 Токенов с logprobs: 1889


#### Старые промпты

In [ ]:
# @title начальный
# Настройка промптов
system_prompt = '''Ты — помощник врача-рентгенолога. Твоя задача — составить структурированное медицинское резюме на основе предоставленной информации о пациенте для подготовки к КТ органов брюшной полости.

ВАЖНО: Верни ответ ТОЛЬКО в формате валидного JSON без markdown-обёрток (без ```json).
Используй цитирование медицинской карты, не выдумывай данные, если информации в тексте нет, пиши "Не указано", не включай информацию, не относящуюся к брюшной полости, если она не влияет на интерпретацию КТ.
Структура JSON (обязательные ключи):
{
  "пациент": "ФИО, возраст, рост, вес",
  "жалобы": "Только жалобы, ставшие причиной выполнения КТ",
  "анамнез_заболевания": "Дата начала, динамика, предварительные диагнозы, лечение",
  "анамнез_жизни": {
    "хронические_заболевания": [],
    "вредные_привычки": [],
    "операции_травмы": [],
    "аллергии": []
  },
  "лабораторные_данные": "Только значимые отклонения и актуальные показатели",
  "инструментальные_данные": "Результаты предыдущих КТ, УЗИ, ЭКГ, важные для сравнения",
  "цель_исследования": "Краткий вывод, что нужно подтвердить или исключить"
}
'''

user_prompt = "Суммаризируй медицинский текст:"

# Формирование сообщений
messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": f"{user_prompt}\n\n{file_content}"
            }
        ]
    }
]

In [ ]:
# @title качественный, но сложный

# Настройка промптов
system_prompt = '''
Ты — помощник врача-рентгенолога для подготовки к КТ органов брюшной полости (ОБП).
Твоя задача — составить структурированное медицинское резюме на основе предоставленной информации о пациенте из ЭМК.
Ты не должен писать свои комментарии, должен выводить и суммировать только важную для врача информацию, но структурировать ее понятно и красиво.
ВАЖНО: Верни ответ ТОЛЬКО в формате чистого JSON текста, не используй ``` или любые другие маркеры форматирования.
Используй дату самого последнего документа в медицинской карте как точку отсчёта для фильтрации данных за 1 год.
Если дат нет — включай все данные.
Все даты приводить к формату ISO 8601: YYYY-MM-DD.
Цитируй блоки медицинской карты, если указываешь диагноз/показатель, добавляй дату документа в скобках: '... (выписка от YYYY-MM-DD)'.
НЕ выдумывай данные.
Если по какому-либо полю нет данных в исходном тексте — пиши "Не указано".
Если данные конфликтуют — указывай самое последнее по дате.
Если текст пустой — верни JSON с полями 'Не указано'."
Включай ВСЕ хронические заболевания и аллергии (для общей оценки рисков),НО в секции 'на_что_обратить_внимание' фокусируйся только на патологиях,
влияющих на интерпретацию КТ ОБП.
Структура JSON (обязательные ключи):
{
  "пациент": "ФИО, возраст, рост, вес(последняя запись)",
  "жалобы": "Только жалобы, ставшие причиной выполнения КТ",
  "анамнез_заболевания": "Дата начала, динамика, предварительные диагнозы, лечение",
  "анамнез_жизни": {
    "хронические_заболевания": [
    "Название заболевания (дата начала: YYYY-MM-DD)"], // Включать ВСЕ хронические заболевания, независимо от давности
    "история болезни": то, чем человек болел, но это не стало хроническим состоянием [],
    "вредные_привычки": [],
    "операции_травмы": [],
    "аллергии": указывать всегда все записи в независимости от даты[]
  },
  "лабораторные_данные": [
  {"показатель": "...", "значение": "...", "дата": "YYYY-MM-DD"}], // Только значимые отклонения за последний год
  "инструментальные_данные": [
  {"исследование": "...", "дата": "YYYY-MM-DD", "находки": "..."}], // Важные для сравнения при КТ ОБП
  "цель_исследования": "Краткий вывод, что нужно подтвердить или исключить",
  "на_что_обратить_внимание_для_задачи_КТ_ОБП": [
  "пункт 1",
  "пункт 2"] // Только патологии, влияющие на интерпретацию КТ ОБП
}
'''

user_prompt = "Суммаризируй медицинский текст:"

---
# **Судья**

In [45]:
import sys
import subprocess

def install_package(package):
    """Устанавливает пакет, если он не установлен"""
    try:
        __import__(package.replace("-", "_"))
    except ImportError:
        print(f"Устанавливаю {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package} установлен")

# Устанавливаем необходимые пакеты
packages = ["rouge-score", "bert-score", "chardet", "openai", "tqdm", "pandas", "openpyxl", "numpy"]
for pkg in packages:
    install_package(pkg)


Устанавливаю rouge-score...
✓ rouge-score установлен
Устанавливаю bert-score...
✓ bert-score установлен


In [50]:
# @title Конфигич
file_path = "/content/text1_source.txt"
output_path = "/content/medical_summary.json"
packages_path = "/content/summary"
result_path = "/content/results"
temperature = 0.2
max_tokens = 2000
enable_thinking = False

In [54]:
import pandas as pd
import numpy as np
import json
import time
import re
from pathlib import Path
from openai import OpenAI
from tqdm import tqdm
import chardet

# ============================================================================
# КОНФИГУРАЦИЯ (Взято из конфигов выше)
# ============================================================================

# Explicitly define variables from the config cell to ensure scope
file_path = "/content/text1_source.txt"
output_path = "/content/medical_summary.json"
packages_path = "/content/summary"
result_path = "/content/results"

VLLM_BASE_URL = "https://vllm.ru.tuna.am/v1"
VLLM_API_KEY = "" # Ensure this is correctly set if needed

JUDGE_MODEL = "Qwen/Qwen3.5-9B"

TEMPERATURE = 0.0
MAX_TOKENS = 8192
MAX_RETRIES = 3
DELAY_BETWEEN_CALLS = 1.0

WEIGHTS = {
    "factual_accuracy": 0.40,
    "completeness": 0.30,
    "clinical_relevance": 0.20,
    "structure": 0.10
}

# Use the result_path from the config for the judge's output directory
OUTPUT_DIR = Path(result_path)

# ============================================================================
# ФУНКЦИИ ЗАГРУЗКИ
# ============================================================================

def detect_encoding(file_path_str): # Renamed argument to avoid conflict with global file_path
    with open(file_path_str, 'rb') as f:
        raw_data = f.read(500000)
        result = chardet.detect(raw_data)
        return result['encoding'] if result['encoding'] else 'utf-8'

def load_text(file_path_str): # Renamed argument
    if not file_path_str or not Path(file_path_str).exists():
        return ""
    try:
        with open(file_path_str, 'r', encoding='utf-8') as f:
            return f.read().strip()
    except UnicodeDecodeError:
        try:
            encoding = detect_encoding(file_path_str)
            with open(file_path_str, 'r', encoding=encoding) as f:
                return f.read().strip()
        except:
            try:
                with open(file_path_str, 'rb') as f:
                    return f.read().decode('utf-8', errors='ignore').strip()
            except:
                return ""

# ============================================================================
# ПРОМПТ ДЛЯ LLM-СУДЬИ
# ============================================================================

JUDGE_PROMPT_FULL = """Ты — эксперт-рентгенолог с большим опытом. Твоя задача — оценить качество медицинского резюме (суммаризации), которое было создано для врача-рентгенолога, который на данный момент смотрит КТ ОБП (ОРГАНЫ БРЮШНОЙ ПОЛОСТИ) пациента, суммаризацию карты которого я приложила.
*Важно*: есть несущественные и неважные пропуски, а есть очень критичные, оценивай это дополнительно при сниженииии ошибок, например, если у человека есть аллергия в эмк, но нет в суммаризации - это плохо
**Исходная электронная медицинская карта (ЭМК) пациента:**
{source_text}

**Резюме для врача-рентгенолога (суммаризация):**
{summary_text}

**Критерии оценки (каждый от 0 до 10):**

1. **Фактологическая точность (0-10):**
   - Оцени, насколько информация в резюме соответствует исходной ЭМК
   - Нет ли выдуманных фактов (галлюцинаций)
   - Нет ли противоречий с исходным текстом

2. **Полнота (0-10):**
   - Оцени, насколько резюме охватывает ключевую информацию для рентгенолога
   - Должны быть отражены: жалобы, анамнез заболевания, анамнез жизни, лабораторные данные, инструментальные исследования

3. **Клиническая релевантность (0-10):**
   - Оцени, насколько информация полезна для врача-рентгенолога при описании КТ органов брюшной полости

4. **Структурированность (0-10):**
   - Оцени, насколько резюме следует требуемой схеме из 5 разделов

**Ответ напиши строго в следующем формате (без лишних слов, только оценки и краткий комментарий):**
Фактологическая точность: [число от 0 до 10]
Полнота: [число от 0 до 10]
Клиническая релевантность: [число от 0 до 10]
Структурированность: [число от 0 до 10]
Комментарий: [текст с цитатами из основной ЭМК, объясняющий причину снижения оценок]
"""

def parse_scores(text):
    """Парсит оценки из текста ответа судьи"""
    scores = {
        "factual_accuracy": 0,
        "completeness": 0,
        "clinical_relevance": 0,
        "structure": 0,
        "comment": ""
    }

    # Паттерны для оценок (ищем в первых 1000 символах)
    first_part = text[:1000]

    patterns = {
        "factual_accuracy": r'Фактологическая точность:\s*(\d+(?:\.\d+)?)',
        "completeness": r'Полнота:\s*(\d+(?:\.\d+)?)',
        "clinical_relevance": r'Клиническая релевантность:\s*(\d+(?:\.\d+)?)',
        "structure": r'Структурированность:\s*(\d+(?:\.\d+)?)',
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, first_part, re.IGNORECASE)
        if match:
            try:
                scores[key] = min(10, max(0, float(match.group(1))))
            except:
                scores[key] = 0

    # Парсинг комментария
    comment_match = re.search(
        r'Комментарий:\s*(.+?)(?=\n\*\*|\n\d+\.\s|\Z)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    if not comment_match:
        comment_match = re.search(
            r'Комментарий:\s*(.+?)(?=\n\n|\n[А-Я]|$)',
            text,
            re.IGNORECASE | re.DOTALL
        )

    if comment_match:
        comment = comment_match.group(1).strip()
        comment = re.sub(r'\n\s*\n', '\n', comment)
        scores["comment"] = comment[:2000] if comment else ""
    else:
        lines = text.strip().split('\n')
        for i, line in enumerate(lines):
            if 'Комментарий:' in line:
                comment = line.split('Комментарий:', 1)[1].strip()
                if len(comment) < 100 and i + 1 < len(lines):
                    comment += ' ' + lines[i + 1].strip()
                scores["comment"] = comment[:2000]
                break

    return scores

def evaluate_summary(client, source_text, summary_text, case_id, model_id, log_dir):
    """Оценивает суммаризацию с помощью LLM-судьи"""

    prompt = JUDGE_PROMPT_FULL.format(
        source_text=source_text if source_text else "Нет данных",
        summary_text=summary_text if summary_text else "Нет данных"
    )

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                extra_body={
                    "chat_template_kwargs": {"enable_thinking": False}  # Отключение thinking режима
                }
            )

            content = response.choices[0].message.content

            # Логируем только ответ судьи
            log_file = log_dir / f"log_case{case_id}_{model_id.replace(' ', '_')}.txt"
            with open(log_file, 'w', encoding='utf-8') as f:
                f.write(content)

            scores = parse_scores(content)

            total = sum([scores[k] for k in ["factual_accuracy", "completeness",
                                              "clinical_relevance", "structure"]])

            if total == 0:
                print(f"  ⚠️ Парсинг не удался для case {case_id}, model {model_id}")
                # If parsing fails, return default scores without fallback to classic metrics
                scores = {
                    "factual_accuracy": 0, "completeness": 0,
                    "clinical_relevance": 0, "structure": 0,
                    "comment": "JSON parse failed"
                }

            for field in ["factual_accuracy", "completeness", "clinical_relevance", "structure"]:
                scores[f"{field}_norm"] = scores[field] / 10.0

            return scores

        except Exception as e:
            print(f"Ошибка {case_id}/{model_id} (попытка {attempt+1}): {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(2 ** attempt)
            else:
                # If API call fails, return default scores without fallback to classic metrics
                return {
                    "factual_accuracy": 0,
                    "completeness": 0,
                    "clinical_relevance": 0,
                    "structure": 0,
                    "factual_accuracy_norm": 0,
                    "completeness_norm": 0,
                    "clinical_relevance_norm": 0,
                    "structure_norm": 0,
                    "comment": "API call failed"
                }

# ============================================================================
# ОСНОВНОЙ КОД
# ============================================================================

print("=" * 60)
print("Оценка медицинских суммаризаций с LLM-судьёй")
print(f"Модель-судья: {JUDGE_MODEL}")
print("Режим thinking: ОТКЛЮЧЁН")
print("=" * 60)

# 1. Подготовка
client = OpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
log_dir = OUTPUT_DIR / "judge_logs"
log_dir.mkdir(exist_ok=True)

# 2. Загрузка данных и оценка суммаризаций из папки
print("\n1. Загрузка эталонного текста и оценка суммаризаций из папки...")

reference_text = load_text(file_path) # Use global file_path from config
if not reference_text:
    raise FileNotFoundError(f"Эталонный файл '{file_path}' не найден или пуст.")

summary_files = list(Path(packages_path).glob('*.txt')) + \
                list(Path(packages_path).glob('*.json'))

if not summary_files:
    print(f"⚠️ В папке '{packages_path}' не найдено файлов для суммаризации.")
else:
    print(f"   Найдено {len(summary_files)} файлов суммаризаций в '{packages_path}'")
    print(f"   Эталонный файл: '{file_path}'")

results = []
with tqdm(total=len(summary_files), desc="Оценка суммаризаций") as pbar:
    for summary_file in summary_files:
        llm_name = summary_file.stem
        summary_text = load_text(summary_file)

        if not summary_text:
            scores = {
                "factual_accuracy": 0, "completeness": 0,
                "clinical_relevance": 0, "structure": 0,
                "factual_accuracy_norm": 0, "completeness_norm": 0,
                "clinical_relevance_norm": 0, "structure_norm": 0,
                "comment": "Пустая суммаризация"
            }
        else:
            scores = evaluate_summary(
                client, reference_text, summary_text,
                llm_name, llm_name, log_dir # case_id and model_id can be the same here
            )
            time.sleep(DELAY_BETWEEN_CALLS)

        weighted_score = (
            WEIGHTS["factual_accuracy"] * scores["factual_accuracy_norm"] +
            WEIGHTS["completeness"] * scores["completeness_norm"] +
            WEIGHTS["clinical_relevance"] * scores["clinical_relevance_norm"] +
            WEIGHTS["structure"] * scores["structure_norm"]
        )

        results.append({
            "summary_file": summary_file.name,
            "llm_id": llm_name,
            "factual_accuracy": scores.get("factual_accuracy", 0),
            "completeness": scores.get("completeness", 0),
            "clinical_relevance": scores.get("clinical_relevance", 0),
            "structure": scores.get("structure", 0),
            "weighted_score": weighted_score,
            "comment": scores.get("comment", "")
        })
        pbar.update(1)

# 3. Сохранение результатов
print("\n2. Сохранение результатов...")

if results:
    results_df = pd.DataFrame(results)
    detailed_output_path = OUTPUT_DIR / "detailed_results.xlsx"
    results_df.to_excel(detailed_output_path, index=False)
    results_df.to_csv(OUTPUT_DIR / "detailed_results.csv", index=False, encoding='utf-8-sig')

    agg_df = results_df.groupby("llm_id").agg({
        "weighted_score": ["mean", "std", "min", "max", "count"],
        "factual_accuracy": "mean",
        "completeness": "mean",
        "clinical_relevance": "mean",
        "structure": "mean"
    }).round(4)
    agg_df.columns = ["mean_score", "std_score", "min_score", "max_score", "n_samples",
                      "factual_accuracy", "completeness", "clinical_relevance", "structure"]
    agg_df = agg_df.sort_values("mean_score", ascending=False)
    agg_df["rank"] = range(1, len(agg_df) + 1)
    aggregated_output_path = OUTPUT_DIR / "aggregated_results.xlsx"
    agg_df.to_excel(aggregated_output_path)

    submission = results_df[["summary_file", "llm_id", "weighted_score"]].copy()
    submission.columns = ["id", "llm_id", "total_score"]
    submission_output_path = OUTPUT_DIR / "submission.xlsx"
    submission.to_excel(submission_output_path, index=False)

    print("\n" + "=" * 60)
    print("РЕЗУЛЬТАТЫ ПО МОДЕЛЯМ:")
    print("=" * 60)
    print(agg_df[["mean_score", "factual_accuracy", "completeness",
                  "clinical_relevance", "structure", "rank"]].to_string())

    print("\n3. Статистика оценок:")
    print(f"   Всего оценено: {len(results_df)} суммаризаций")
    print(f"   Средняя оценка: {results_df['weighted_score'].mean():.4f}")
    print(f"   Медианная оценка: {results_df['weighted_score'].median():.4f}")

    log_files = list(log_dir.glob("*.txt"))
    print(f"   Сохранено логов: {len(log_files)}")

    print("\n" + "=" * 60)
    print("ГОТОВО! Результаты сохранены в папке:")
    print(OUTPUT_DIR.absolute())
    print("=" * 60)
    print("\nФайлы:")
    print(f"  - {detailed_output_path.name}  (оценки всех суммаризаций)")
    print(f"  - {aggregated_output_path.name} (агрегированные результаты)")
    print(f"  - {submission_output_path.name} (файл для отправки организаторам)")
    print(f"  - judge_logs/ (логи ответов судьи)")
else:
    print("   Нет результатов для сохранения. Возможно, не найдено файлов для суммаризации.")

Оценка медицинских суммаризаций с LLM-судьёй
Модель-судья: Qwen/Qwen3.5-9B
Режим thinking: ОТКЛЮЧЁН

1. Загрузка эталонного текста и оценка суммаризаций из папки...
   Найдено 10 файлов суммаризаций в '/content/summary'
   Эталонный файл: '/content/text1_source.txt'


Оценка суммаризаций:  30%|███       | 3/10 [00:19<00:44,  6.30s/it]

  ⚠️ Парсинг не удался для case text1_model6, model text1_model6


Оценка суммаризаций: 100%|██████████| 10/10 [01:11<00:00,  7.19s/it]


2. Сохранение результатов...

РЕЗУЛЬТАТЫ ПО МОДЕЛЯМ:
                         mean_score  factual_accuracy  completeness  clinical_relevance  structure  rank
llm_id                                                                                                  
text1_source_summary_v2        0.88               9.0           8.0                 9.0       10.0     1
text1_model2                   0.78              10.0           6.0                 5.0       10.0     2
text1_source_summary_v1        0.77               6.0           9.0                 8.0       10.0     3
text1_source_summary_v4        0.70               6.0           7.0                 8.0        9.0     4
text1_source_summary_v3        0.70               6.0           7.0                 8.0        9.0     5
text1_model5                   0.67               8.0           6.0                 4.0        9.0     6
text1_model3                   0.65               8.0           4.0                 6.0        9.0     7
t

---
# **Метрички и сигналки**

### Токен-уровневая оценка неуверенности модели, галлюцинаций

In [ ]:
# @title Тест поддержки logprobs
import openai

client = openai.OpenAI(
    base_url="https://vllm.ru.tuna.am/v1",
    api_key=""  # Ваш ключ
)

try:
    response = client.chat.completions.create(
        model="Qwen/Qwen3.5-9B",
        messages=[{"role": "user", "content": "Тест"}],
        max_tokens=10,
        logprobs=True,
        top_logprobs=5
    )

    if response.choices[0].logprobs and response.choices[0].logprobs.content:
        print("✅ logprobs ПОДДЕРЖИВАЕТСЯ!")
        print(f"   Токенов с logprobs: {len(response.choices[0].logprobs.content)}")
        print(f"   Пример первого токена: {response.choices[0].logprobs.content[0]}")
    else:
        print("⚠️ logprobs не вернулись (возможно, отключены на сервере)")

except Exception as e:
    print(f"❌ Ошибка: {e}")
    print("   Скорее всего, эндпоинт не поддерживает logprobs")

✅ logprobs ПОДДЕРЖИВАЕТСЯ!
   Токенов с logprobs: 10
   Пример первого токена: ChatCompletionTokenLogprob(token='Thinking', bytes=[84, 104, 105, 110, 107, 105, 110, 103], logprob=-0.26395514607429504, top_logprobs=[TopLogprob(token='Thinking', bytes=[84, 104, 105, 110, 107, 105, 110, 103], logprob=-0.26395514607429504), TopLogprob(token='Okay', bytes=[79, 107, 97, 121], logprob=-2.1389551162719727), TopLogprob(token='The', bytes=[84, 104, 101], logprob=-2.3889551162719727), TopLogprob(token='Here', bytes=[72, 101, 114, 101], logprob=-4.763955116271973), TopLogprob(token='П', bytes=[208, 159], logprob=-6.076455116271973)])


In [34]:
# @title УНИВЕРСАЛЬНАЯ система медицинских сущностей

import re
from typing import Dict, List, Set, Optional
from dataclasses import dataclass, field
from pathlib import Path
import json

# ============================================
# УРОВЕНЬ 1: БАЗОВЫЕ ПАТТЕРНЫ (универсальные)
# ============================================

@dataclass
class EntityPattern:
    name: str
    regex: str
    critical: bool
    specialty: str = "general"  # general, cardiology, neurology, etc.

# ✅ Базовые паттерны — работают для ЛЮБОЙ медицинской карты
BASE_ENTITY_PATTERNS = [
    # Препараты: любое слово + дозировка
    EntityPattern("препарат",
                  r"\b([А-Я][а-я]+(?:\s+[А-Я][а-я]+){0,2}\s+\d+\s*(?:мг|мкг|г|мл|капс|таб|амп|фл))\b",
                  critical=True),

    # Дозировки отдельно (для анализа единиц измерения)
    EntityPattern("дозировка",
                  r"\b(\d+\s*(?:мг|мкг|г|мл|капс|таб|амп|фл|ЕД|МЕ))\b",
                  critical=True),

    # Диагнозы: более гибкий паттерн
    EntityPattern("диагноз",
                  r"\b((?:Хронический|Острый|Обострение|Первичный|Вторичный|Тяжелый|Легкий|Средней тяжести)\s+[А-Я][а-я]+(?:\s+[А-Я][а-я]+){0,4}(?:\s+(?:болезни|синдром|недостаточность|дисфункция|поражение))?)\b",
                  critical=True),

    # Лабораторные показатели: аббревиатуры + названия
    EntityPattern("лабораторный_показатель",
                  r"\b((?:[А-Я]{2,5}|[А-Я][а-я]+\s+[А-Я][а-я]+)\s*(?::\s*\d+\.?\d*\s*(?:ммоль|мг|ЕД|нг|мкг)?/?(?:л|мл|ч)?)?)\b",
                  critical=True),

    # Анатомические структуры: корни слов + окончания
    EntityPattern("анатомическая_структура",
                  r"\b((?:печен|поджелудочн|желочн|селезён|почк|кишечн|желудк|лёгк|сердц|аорт|диафрагм|средостен|предсерд|желудочк|клапан|сосуд|артери|вен)[а-я]*\s*(?:[а-я]+){0,2})\b",
                  critical=True),

    # Исследования: любые медицинские аббревиатуры
    EntityPattern("исследование",
                  r"\b((?:КТ|МРТ|УЗИ|ЭКГ|ФГДС|Рентген|КОР|ПЭТ|Сцинтиграфия|ЭхоКГ|Холтер|СМАД)\s*(?:[А-Я]{2,4}|[а-я]+){0,3})\b",
                  critical=False),

    # Даты (для проверки временных рамок)
    EntityPattern("дата",
                  r"\b(\d{4}-\d{2}-\d{2}|\d{2}\.\d{2}\.\d{4})\b",
                  critical=False),
]

# ============================================
# УРОВЕНЬ 2: СПЕЦИАЛЬНОСТИ (подключаемые модули)
# ============================================

SPECIALTY_PATTERNS = {
    "cardiology": [
        EntityPattern("кардио_диагноз",
                      r"\b((?:ИБС|АГ|АГБ|Сердечная недостаточность|Аритмия|Фибрилляция предсердий|Стенокардия|Инфаркт)\s*(?:[а-я]+){0,3})\b",
                      critical=True, specialty="cardiology"),
        EntityPattern("кардио_показатель",
                      r"\b((?:АД|САД|ДАД|ЧСС|ФВ|ЛЖ|ПЖ|КДО|КСО)\s*(?::\s*\d+\.?\d*)?)\b",
                      critical=True, specialty="cardiology"),
    ],

    "neurology": [
        EntityPattern("невро_диагноз",
                      r"\b((?:ОНМК|Инсульт|ТИА|Деменция|Болезнь [А-Я][а-я]+|Невропатия|Эпилепсия)\s*(?:[а-я]+){0,3})\b",
                      critical=True, specialty="neurology"),
    ],

    "gastroenterology": [
        EntityPattern("гастро_диагноз",
                      r"\b((?:Гастрит|Язва|Панкреатит|Холецистит|Гепатит|Цирроз|Колит)\s*(?:[а-я]+){0,3})\b",
                      critical=True, specialty="gastroenterology"),
    ],

    "oncology": [
        EntityPattern("онко_диагноз",
                      r"\b((?:Рак|Карцинома|Саркома|Лимфома|Меланома|Опухоль|Новообразование|T[1-4]N[0-3]M[0-1])\s*(?:[а-я]+){0,3})\b",
                      critical=True, specialty="oncology"),
    ],
}

# ============================================
# УРОВЕНЬ 3: СТОП-СЛОВА (только структурные!)
# ============================================

# ✅ ТОЛЬКО структурные слова JSON и промпта
STRUCTURAL_STOP_WORDS = {
    # JSON-ключи
    'пациент', 'жалобы', 'анамнез_заболевания', 'анамнез_жизни',
    'хронические_заболевания', 'история болезни', 'вредные привычки',
    'операции', 'травмы', 'аллергии', 'лабораторные_данные',
    'инструментальные_данные', 'цель_исследования', 'на_что_обратить_внимание',
    'показатель', 'значение', 'дата', 'исследование', 'находки',

    # Слова из промпта (инструкции модели)
    'фильтруй', 'включай', 'указывать', 'возвращай', 'используй', 'пиши',
    'если', 'нет', 'данных', 'не', 'указано', 'только', 'важные',
    'для', 'сравнения', 'при', 'за', 'последний', 'актуальные',
    'результаты', 'предыдущих', 'краткий', 'вывод', 'что', 'нужно',
    'подтвердить', 'или', 'исключить', 'пункт', 'задачи', 'кт', 'обп',

    # Общие слова (не медицинские)
    'мужчина', 'женщина', 'возраст', 'лет', 'назад', 'отказ', 'более',
    'самостоятельно', 'принимал', 'ежедневно', 'эффектом', 'предварительный',
    'тяжелой', 'степени', 'преимущественно', 'тип', 'начало', 'год', 'годы',
    'с', 'в', 'на', 'от', 'до', 'и', 'или', 'как', 'причину', 'после',
    'приема', 'пищи', 'тошнота', 'общая', 'слабость', 'боли', 'животе',
}

# ❌ УДАЛИТЬ из стоп-слов (это реальные медицинские термины!)
# 'киста', 'эмфизема', 'пневмония', 'панкреатит', 'холецистит',
# 'ХОБЛ', 'атеросклероз', 'гипертоническая', 'ишемическая'

# ============================================
# КЛАСС ДЛЯ УПРАВЛЕНИЯ ПАТТЕРНАМИ
# ============================================

class MedicalEntityExtractor:
    def __init__(self, specialties: List[str] = None, custom_stop_words: Set[str] = None):
        """
        specialties: список специальностей для подключения паттернов
        custom_stop_words: дополнительные стоп-слова
        """
        self.base_patterns = BASE_ENTITY_PATTERNS
        self.specialty_patterns = []
        self.stop_words = STRUCTURAL_STOP_WORDS.copy()

        # Подключаем паттерны специальностей
        if specialties:
            for spec in specialties:
                if spec in SPECIALTY_PATTERNS:
                    self.specialty_patterns.extend(SPECIALTY_PATTERNS[spec])

        # Добавляем кастомные стоп-слова
        if custom_stop_words:
            self.stop_words.update(custom_stop_words)

        # Все паттерны вместе
        self.all_patterns = self.base_patterns + self.specialty_patterns

    def get_patterns_for_text(self, text: str) -> List[EntityPattern]:
        """
        Автоматически определяет, какие паттерны применять
        на основе анализа текста
        """
        # Простая эвристика: если в тексте есть ключевые слова специальности
        specialty_keywords = {
            "cardiology": ["сердц", "карди", "давлен", "пульс", "экг", "эхокг"],
            "neurology": ["головн", "мозг", "онмк", "инсульт", "невро"],
            "gastroenterology": ["желуд", "печен", "панкреат", "холецист", "гастр"],
            "oncology": ["опухол", "рак", "карцином", "метастаз", "онко"],
        }

        text_lower = text.lower()
        active_specialties = []

        for spec, keywords in specialty_keywords.items():
            if any(kw in text_lower for kw in keywords):
                active_specialties.append(spec)

        # Если нашли специальность — загружаем её паттерны
        if active_specialties:
            return self.base_patterns + [
                p for spec in active_specialties
                for p in SPECIALTY_PATTERNS.get(spec, [])
            ]

        return self.base_patterns

    def is_false_positive(self, entity_text: str) -> bool:
        """
        Проверка на ложную сущность
        """
        text_lower = entity_text.lower().strip()

        # В стоп-словах
        if text_lower in self.stop_words:
            return True

        # Слишком короткое
        if len(text_lower) < 3:
            return True

        # Содержит JSON-символы
        if any(char in entity_text for char in ['"', ':', '{', '}', '[', ']']):
            return True

        # Только предлоги/союзы
        if text_lower in {'с', 'в', 'на', 'от', 'до', 'и', 'или', 'как', 'за', 'при'}:
            return True

        return False

    def extract_entities(self, text: str, logprobs_: List) -> List[Dict]:
        """
        Извлечение сущностей с применением всех правил
        """
        # Определяем актуальные паттерны
        patterns = self.get_patterns_for_text(text)

        entities = []
        seen_texts: Set[str] = set()

        # Токенизация для поиска позиций
        token_texts = [getattr(t, 'token', '') for t in logprobs_data]

        for pattern in patterns:
            for match in re.finditer(pattern.regex, text, re.IGNORECASE):
                entity_text = match.group(0).strip()

                # Фильтрация ложных сущностей
                if self.is_false_positive(entity_text):
                    continue

                # Удаление дубликатов
                if entity_text in seen_texts:
                    continue
                seen_texts.add(entity_text)

                # Находим токены
                start_char, end_char = match.start(), match.end()
                token_indices = self._find_token_indices(
                    token_texts, start_char, end_char
                )

                if token_indices:
                    entities.append({
                        'type': pattern.name,
                        'text': entity_text,
                        'start_char': start_char,
                        'end_char': end_char,
                        'token_indices': token_indices,
                        'critical': pattern.critical,
                        'specialty': pattern.specialty
                    })

        return entities

    def _find_token_indices(self, token_texts: List[str],
                           start_char: int, end_char: int) -> List[int]:
        """Находит индексы токенов для спана"""
        indices = []
        char_pos = 0

        for i, token_text in enumerate(token_texts):
            token_start = char_pos
            token_end = char_pos + len(token_text)

            if token_start < end_char and token_end > start_char:
                indices.append(i)

            char_pos = token_end

        return indices


# ============================================
# ПРИМЕР ИСПОЛЬЗОВАНИЯ
# ============================================

"""
# Для универсального использования (любая карта)
extractor = MedicalEntityExtractor()

# Для кардиологической карты
extractor = MedicalEntityExtractor(specialties=["cardiology"])

# Для мультидисциплинарной карты
extractor = MedicalEntityExtractor(specialties=["cardiology", "neurology", "gastroenterology"])

# С кастомными стоп-словами
extractor = MedicalEntityExtractor(
    specialties=["cardiology"],
    custom_stop_words={'специфичное_слово_из_вашей_карты'}
)

# Извлечение сущностей
entities = extractor.extract_entities(summary_text, summary_logprobs)
"""

# ============================================
# СОХРАНЕНИЕ/ЗАГРУЗКА КОНФИГУРАЦИИ
# ============================================

def save_extractor_config(extractor: MedicalEntityExtractor, path: str):
    """Сохраняет конфигурацию для повторного использования"""
    config = {
        'stop_words': list(extractor.stop_words),
        'specialties': [p.specialty for p in extractor.specialty_patterns
                       if p.specialty != "general"]
    }
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

def load_extractor_config(path: str) -> MedicalEntityExtractor:
    """Загружает конфигурацию"""
    with open(path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    return MedicalEntityExtractor(
        specialties=config.get('specialties', []),
        custom_stop_words=set(config.get('stop_words', []))
    )

print("✅ Универсальная система медицинских сущностей загружена!")
print(f"   Базовых паттернов: {len(BASE_ENTITY_PATTERNS)}")
print(f"   Специальностей: {len(SPECIALTY_PATTERNS)}")
print(f"   Стоп-слов: {len(STRUCTURAL_STOP_WORDS)}")

✅ Универсальная система медицинских сущностей загружена!
   Базовых паттернов: 7
   Специальностей: 4
   Стоп-слов: 86


In [37]:
# @title Метрики качества суммаризации (УНИВЕРСАЛЬНАЯ ВЕРСИЯ)

import math
import re
import json
from typing import Dict, List, Optional, Set
from dataclasses import dataclass, field
from pathlib import Path
from datetime import datetime

# ============================================
# ЧАСТЬ 1: УНИВЕРСАЛЬНЫЕ ПАТТЕРНЫ
# ============================================

@dataclass
class EntityPattern:
    name: str
    regex: str
    critical: bool
    specialty: str = "general"

# ✅ Базовые паттерны — работают для ЛЮБОЙ медицинской карты
BASE_ENTITY_PATTERNS = [
    # Препараты: любое слово + дозировка
    EntityPattern("препарат",
                  r"\b([А-Я][а-я]+(?:\s+[А-Я][а-я]+){0,2}\s+\d+\s*(?:мг|мкг|г|мл|капс|таб|амп|фл))\b",
                  critical=True),

    # Дозировки отдельно
    EntityPattern("дозировка",
                  r"\b(\d+\s*(?:мг|мкг|г|мл|капс|таб|амп|фл|ЕД|МЕ))\b",
                  critical=True),

    # Диагнозы: гибкий паттерн с заглавной буквы
    EntityPattern("диагноз",
                  r"\b((?:Хронический|Острый|Обострение|Первичный|Вторичный|Тяжелый|Легкий|Средней тяжести|Двусторонний|Диффузный)\s+[А-Я][а-я]+(?:\s+[А-Я][а-я]+){0,4}(?:\s+(?:болезни|синдром|недостаточность|дисфункция|поражение))?)\b",
                  critical=True),

    # Лабораторные показатели: аббревиатуры + названия
    EntityPattern("лабораторный_показатель",
                  r"\b((?:[А-Я]{2,5}|[А-Я][а-я]+\s+[А-Я][а-я]+)\s*(?::\s*\d+\.?\d*\s*(?:ммоль|мг|ЕД|нг|мкг)?/?(?:л|мл|ч)?)?)\b",
                  critical=True),

    # Анатомические структуры: корни слов
    EntityPattern("анатомическая_структура",
                  r"\b((?:печен|поджелудочн|желочн|селезён|почк|кишечн|желудк|лёгк|сердц|аорт|диафрагм|средостен|предсерд|желудочк|клапан|сосуд|артери|вен)[а-я]*\s*(?:[а-я]+){0,2})\b",
                  critical=True),

    # Исследования: медицинские аббревиатуры
    EntityPattern("исследование",
                  r"\b((?:КТ|МРТ|УЗИ|ЭКГ|ФГДС|Рентген|КОР|ПЭТ|Сцинтиграфия|ЭхоКГ|Холтер|СМАД)\s*(?:[А-Я]{2,4}|[а-я]+){0,3})\b",
                  critical=False),
]

# ✅ Паттерны по специальностям (подключаемые модули)
SPECIALTY_PATTERNS = {
    "cardiology": [
        EntityPattern("кардио_диагноз",
                      r"\b((?:ИБС|АГ|АГБ|Сердечная недостаточность|Аритмия|Фибрилляция предсердий|Стенокардия|Инфаркт)\s*(?:[а-я]+){0,3})\b",
                      critical=True, specialty="cardiology"),
        EntityPattern("кардио_показатель",
                      r"\b((?:АД|САД|ДАД|ЧСС|ФВ|ЛЖ|ПЖ|КДО|КСО)\s*(?::\s*\d+\.?\d*)?)\b",
                      critical=True, specialty="cardiology"),
    ],
    "neurology": [
        EntityPattern("невро_диагноз",
                      r"\b((?:ОНМК|Инсульт|ТИА|Деменция|Болезнь [А-Я][а-я]+|Невропатия|Эпилепсия)\s*(?:[а-я]+){0,3})\b",
                      critical=True, specialty="neurology"),
    ],
    "gastroenterology": [
        EntityPattern("гастро_диагноз",
                      r"\b((?:Гастрит|Язва|Панкреатит|Холецистит|Гепатит|Цирроз|Колит)\s*(?:[а-я]+){0,3})\b",
                      critical=True, specialty="gastroenterology"),
    ],
    "oncology": [
        EntityPattern("онко_диагноз",
                      r"\b((?:Рак|Карцинома|Саркома|Лимфома|Меланома|Опухоль|Новообразование)\s*(?:[а-я]+){0,3})\b",
                      critical=True, specialty="oncology"),
    ],
}

# ✅ СТОП-СЛОВА — ТОЛЬКО структурные (не медицинские термины!)
STRUCTURAL_STOP_WORDS = {
    # JSON-ключи
    'пациент', 'жалобы', 'анамнез_заболевания', 'анамнез_жизни',
    'хронические_заболевания', 'история болезни', 'вредные привычки',
    'операции', 'травмы', 'аллергии', 'лабораторные_данные',
    'инструментальные_данные', 'цель_исследования', 'на_что_обратить_внимание',
    'показатель', 'значение', 'дата', 'исследование', 'находки',

    # Слова из промпта (инструкции)
    'фильтруй', 'включай', 'указывать', 'возвращай', 'используй', 'пиши',
    'если', 'нет', 'данных', 'не', 'указано', 'только', 'важные',
    'для', 'сравнения', 'при', 'за', 'последний', 'актуальные',
    'результаты', 'предыдущих', 'краткий', 'вывод', 'что', 'нужно',
    'подтвердить', 'или', 'исключить', 'пункт', 'задачи', 'кт', 'обп',

    # Общие слова (не медицинские)
    'мужчина', 'женщина', 'возраст', 'лет', 'назад', 'отказ', 'более',
    'самостоятельно', 'принимал', 'ежедневно', 'эффектом', 'предварительный',
    'тяжелой', 'степени', 'преимущественно', 'тип', 'начало', 'год', 'годы',
    'с', 'в', 'на', 'от', 'до', 'и', 'или', 'как', 'причину', 'после',
    'приема', 'пищи', 'тошнота', 'общая', 'слабость', 'боли', 'животе',
}

# ============================================
# ЧАСТЬ 2: КЛАСС ДЛЯ ИЗВЛЕЧЕНИЯ СУЩНОСТЕЙ
# ============================================

class MedicalEntityExtractor:
    def __init__(self, specialties: List[str] = None, custom_stop_words: Set[str] = None):
        self.base_patterns = BASE_ENTITY_PATTERNS
        self.specialty_patterns = []
        self.stop_words = STRUCTURAL_STOP_WORDS.copy()

        if specialties:
            for spec in specialties:
                if spec in SPECIALTY_PATTERNS:
                    self.specialty_patterns.extend(SPECIALTY_PATTERNS[spec])

        if custom_stop_words:
            self.stop_words.update(custom_stop_words)

        self.all_patterns = self.base_patterns + self.specialty_patterns

    def detect_specialty(self, text: str) -> List[str]:
        """Авто-определение специальности по тексту"""
        specialty_keywords = {
            "cardiology": ["сердц", "карди", "давлен", "пульс", "экг", "эхокг", "ибс", "аг "],
            "neurology": ["головн", "мозг", "онмк", "инсульт", "невро", "тиа"],
            "gastroenterology": ["желуд", "печен", "панкреат", "холецист", "гастр", "кишечн"],
            "oncology": ["опухол", "рак", "карцином", "метастаз", "онко", "новообразован"],
        }

        text_lower = text.lower()
        detected = []

        for spec, keywords in specialty_keywords.items():
            if any(kw in text_lower for kw in keywords):
                detected.append(spec)

        return detected

    def get_patterns_for_text(self, text: str) -> List[EntityPattern]:
        """Подбирает паттерны под текст"""
        detected_specialties = self.detect_specialty(text)

        if detected_specialties:
            specialty_pats = [
                p for spec in detected_specialties
                for p in SPECIALTY_PATTERNS.get(spec, [])
            ]
            return self.base_patterns + specialty_pats

        return self.base_patterns

    def is_false_positive(self, entity_text: str) -> bool:
        """Проверка на ложную сущность"""
        text_lower = entity_text.lower().strip()

        if text_lower in self.stop_words:
            return True
        if len(text_lower) < 3:
            return True
        if any(char in entity_text for char in ['"', ':', '{', '}', '[', ']']):
            return True
        if text_lower in {'с', 'в', 'на', 'от', 'до', 'и', 'или', 'как', 'за', 'при'}:
            return True

        return False

    def _find_token_indices(self, token_texts: List[str],
                           start_char: int, end_char: int) -> List[int]:
        """Находит индексы токенов для спана"""
        indices = []
        char_pos = 0

        for i, token_text in enumerate(token_texts):
            token_start = char_pos
            token_end = char_pos + len(token_text)

            if token_start < end_char and token_end > start_char:
                indices.append(i)

            char_pos = token_end

        return indices

    def extract_entities(self, text: str, logprobs_: List) -> List[Dict]:
        """Извлечение сущностей с метриками"""
        patterns = self.get_patterns_for_text(text)
        token_texts = [getattr(t, 'token', '') for t in logprobs_data]

        entities = []
        seen_texts: Set[str] = set()

        for pattern in patterns:
            for match in re.finditer(pattern.regex, text, re.IGNORECASE):
                entity_text = match.group(0).strip()

                if self.is_false_positive(entity_text):
                    continue
                if entity_text in seen_texts:
                    continue
                seen_texts.add(entity_text)

                start_char, end_char = match.start(), match.end()
                token_indices = self._find_token_indices(
                    token_texts, start_char, end_char
                )

                if token_indices:
                    entities.append({
                        'type': pattern.name,
                        'text': entity_text,
                        'start_char': start_char,
                        'end_char': end_char,
                        'token_indices': token_indices,
                        'critical': pattern.critical,
                        'specialty': pattern.specialty
                    })

        return entities


# ============================================
# ЧАСТЬ 3: КЛАСС ДЛЯ РАСЧЁТА МЕТРИК (по CSV)
# ============================================

class TokenUncertaintyMetrics:
    def __init__(self, logprobs_: List, full_text: str):
        self.tokens = logprobs_
        self.token_texts = [getattr(t, 'token', '') for t in logprobs_]
        self.full_text = full_text
        self.extractor = MedicalEntityExtractor()

    def _logprob_to_prob(self, logprob: float) -> float:
        """Конвертация logprob в вероятность"""
        if logprob is None:
            return 0.0
        try:
            return math.exp(logprob)
        except (OverflowError, ValueError):
            return 0.0

    def _calculate_entropy(self, top_logprobs: List) -> float:
        """
        Нормализованная энтропия: H = -Σ P(i) * log(P(i))
        Согласно CSV: нормализуем по log2(N)
        """
        if not top_logprobs:
            return 0.0

        probs = []
        for item in top_logprobs:
            logprob = getattr(item, 'logprob', None)
            if logprob is not None:
                prob = self._logprob_to_prob(logprob)
                if prob > 0:
                    probs.append(prob)

        if not probs:
            return 0.0

        total = sum(probs)
        if total > 0:
            probs = [p / total for p in probs]

        entropy = -sum(p * math.log(p) for p in probs if p > 0)
        max_entropy = math.log2(len(probs)) if len(probs) > 1 else 1.0
        normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0.0

        return min(normalized_entropy, 1.0)

    def _calculate_margin(self, top_logprobs: List) -> float:
        """
        Margin = P1 - P2
        Согласно CSV: отрицательный margin = критический риск
        """
        if len(top_logprobs) < 2:
            return 1.0

        p1 = self._logprob_to_prob(getattr(top_logprobs[0], 'logprob', None))
        p2 = self._logprob_to_prob(getattr(top_logprobs[1], 'logprob', None))

        return p1 - p2

    def calculate_entity_metrics(self, entity: Dict) -> Dict:
        """
        Расчёт 3 метрик для сущности
        Согласно CSV:
        - Prob: MIN по токенам
        - Entropy: MAX по токенам
        - Margin: MIN по токенам
        """
        token_indices = entity['token_indices']

        if not token_indices:
            return {
                'entity': entity['text'],
                'type': entity['type'],
                'critical': entity['critical'],
                'specialty': entity.get('specialty', 'general'),
                'min_prob': None,
                'max_entropy': None,
                'min_margin': None,
                'risk_level': 'UNKNOWN',
                'token_count': 0
            }

        probs = []
        entropies = []
        margins = []

        for idx in token_indices:
            token_data = self.tokens[idx]
            logprob = getattr(token_data, 'logprob', None)
            top_logprobs = getattr(token_data, 'top_logprobs', [])

            if logprob is not None:
                probs.append(self._logprob_to_prob(logprob))

            if top_logprobs:
                entropies.append(self._calculate_entropy(top_logprobs))
                margins.append(self._calculate_margin(top_logprobs))

        # ✅ АГРЕГАЦИЯ по CSV
        min_prob = min(probs) if probs else None
        max_entropy = max(entropies) if entropies else None
        min_margin = min(margins) if margins else None

        risk_level = self._assess_risk(min_prob, max_entropy, min_margin, entity['critical'])

        return {
            'entity': entity['text'],
            'type': entity['type'],
            'critical': entity['critical'],
            'specialty': entity.get('specialty', 'general'),
            'min_prob': round(min_prob, 4) if min_prob else None,
            'max_entropy': round(max_entropy, 4) if max_entropy else None,
            'min_margin': round(min_margin, 4) if min_margin else None,
            'risk_level': risk_level,
            'token_count': len(token_indices)
        }

    def _assess_risk(self, prob: Optional[float], entropy: Optional[float],
                     margin: Optional[float], critical: bool) -> str:
        """Оценка риска по порогам из CSV"""
        if prob is None or entropy is None or margin is None:
            return 'UNKNOWN'

        # 🔴 КРИТИЧЕСКИЙ: отрицательный margin
        if margin < 0:
            return '🔴 HIGH'

        # Пороги из CSV таблицы
        if critical:
            prob_thresholds = (0.9, 0.6)
            margin_thresholds = (0.3, 0.1)
            entropy_thresholds = (0.5, 0.6)
        else:
            prob_thresholds = (0.9, 0.5)
            margin_thresholds = (0.3, 0.1)
            entropy_thresholds = (0.6, 0.8)

        if (
            prob < prob_thresholds[1] or
            margin < margin_thresholds[1] or
            entropy > entropy_thresholds[1]
        ):
            return '🔴 HIGH'

        if (
            prob < prob_thresholds[0] or
            margin < margin_thresholds[0] or
            entropy > entropy_thresholds[0]
        ):
            return '🟡 MEDIUM'

        return '🟢 LOW'

    def get_full_report(self) -> Dict:
        """Полный отчёт"""
        entities = self.extractor.extract_entities(self.full_text, self.tokens)
        entity_metrics = [self.calculate_entity_metrics(e) for e in entities]

        high_risk = [e for e in entity_metrics if e['risk_level'] == '🔴 HIGH']
        medium_risk = [e for e in entity_metrics if e['risk_level'] == '🟡 MEDIUM']
        low_risk = [e for e in entity_metrics if e['risk_level'] == '🟢 LOW']
        critical_high_risk = [e for e in high_risk if e['critical']]
        negative_margin = [e for e in entity_metrics
                          if e['min_margin'] is not None and e['min_margin'] < 0]

        return {
            'total_entities': len(entity_metrics),
            'high_risk_count': len(high_risk),
            'medium_risk_count': len(medium_risk),
            'low_risk_count': len(low_risk),
            'critical_high_risk_count': len(critical_high_risk),
            'negative_margin_count': len(negative_margin),
            'critical_high_risk': critical_high_risk,
            'negative_margin_entities': negative_margin,
            'all_metrics': entity_metrics,
            'detected_specialties': self.extractor.detect_specialty(self.full_text)
        }


# ============================================
# ЧАСТЬ 4: ЗАПУСК АНАЛИЗА
# ============================================

print("🔍 Анализ неопределённости модели...\n")
print("=" * 70)

try:
    metrics_analyzer = TokenUncertaintyMetrics(summary_logprobs, summary_text)
    report = metrics_analyzer.get_full_report()
except NameError:
    print("❌ Ошибка: переменные summary_logprobs и summary_text не найдены!")
    print("   Убедитесь, что ячейка 15 выполнена с logprobs=True")
    raise

# ============================================
# ЧАСТЬ 5: ВЫВОД РЕЗУЛЬТАТОВ
# ============================================

print("📊 ОТЧЁТ ПО МЕТРИКАМ НЕОПРЕДЕЛЁННОСТИ")
print("=" * 70)
print(f"Всего сущностей: {report['total_entities']}")
print(f"🟢 Низкий риск: {report['low_risk_count']}")
print(f"🟡 Средний риск: {report['medium_risk_count']}")
print(f"🔴 Высокий риск: {report['high_risk_count']}")
print(f"⚠️  Отрицательный Margin: {report['negative_margin_count']}")
print(f"🏥 Определённые специальности: {', '.join(report['detected_specialties']) or 'общая'}")
print()

# 🔴 ПРИОРИТЕТ 1: Отрицательный Margin
if report['negative_margin_entities']:
    print("🚨 ТРЕБУЮТ СРОЧНОЙ ПРОВЕРКИ (отрицательный Margin):")
    print("-" * 70)
    for entity in report['negative_margin_entities'][:10]:
        prob_val = f"{entity['min_prob']:.3f}" if entity['min_prob'] is not None else "N/A"
        ent_val = f"{entity['max_entropy']:.3f}" if entity['max_entropy'] is not None else "N/A"
        mar_val = f"{entity['min_margin']:.3f}" if entity['min_margin'] is not None else "N/A"
        print(f"  • {entity['entity']} ({entity['type']})")
        print(f"    Prob: {prob_val} | Entropy: {ent_val} | Margin: {mar_val}")
    if len(report['negative_margin_entities']) > 10:
        print(f"  ... и ещё {len(report['negative_margin_entities']) - 10}")
    print()

# 🔴 ПРИОРИТЕТ 2: Критические сущности высокого риска
if report['critical_high_risk']:
    print("⚠️  КРИТИЧЕСКИЕ СУЩНОСТИ ВЫСОКОГО РИСКА:")
    print("-" * 70)
    for entity in report['critical_high_risk'][:10]:
        prob_val = f"{entity['min_prob']:.3f}" if entity['min_prob'] is not None else "N/A"
        ent_val = f"{entity['max_entropy']:.3f}" if entity['max_entropy'] is not None else "N/A"
        mar_val = f"{entity['min_margin']:.3f}" if entity['min_margin'] is not None else "N/A"
        print(f"  • {entity['entity']} ({entity['type']})")
        print(f"    Prob: {prob_val} | Entropy: {ent_val} | Margin: {mar_val}")
    if len(report['critical_high_risk']) > 10:
        print(f"  ... и ещё {len(report['critical_high_risk']) - 10}")
    print()
else:
    print("✅ Нет критических сущностей высокого риска")
    print()

# ============================================
# ЧАСТЬ 6: СОХРАНЕНИЕ В JSON
# ============================================

high_risk_percentage = round(report['high_risk_count'] / max(report['total_entities'], 1) * 100, 2)

summary_result['uncertainty_metrics'] = {
    'summary': {
        'total_entities': report['total_entities'],
        'high_risk_count': report['high_risk_count'],
        'medium_risk_count': report['medium_risk_count'],
        'low_risk_count': report['low_risk_count'],
        'critical_high_risk_count': report['critical_high_risk_count'],
        'negative_margin_count': report['negative_margin_count'],
        'high_risk_percentage': high_risk_percentage,
        'detected_specialties': report['detected_specialties']
    },
    'priority_review': {
        'negative_margin': report['negative_margin_entities'],
        'critical_high_risk': report['critical_high_risk']
    },
    'all_entities': report['all_metrics']
}

with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_result, f, ensure_ascii=False, indent=2)

print(f"✅ Метрики сохранены в {summary_path}")
print("=" * 70)

# ============================================
# ЧАСТЬ 7: ТАБЛИЦА ТОП-20
# ============================================

print("\n📋 ТОП-20 СУЩНОСТЕЙ ПО РИСКУ:")
print("-" * 70)
print(f"{'#':<3} {'Сущность':<35} {'Тип':<18} {'Риск':<10} {'Prob':<7} {'Ent':<7} {'Mar':<7}")
print("-" * 70)

sorted_entities = sorted(report['all_metrics'],
                         key=lambda x: (x['min_margin'] if x['min_margin'] else 1,
                                       0 if x['risk_level'] == '🔴 HIGH' else 1,
                                       0 if x['critical'] else 1))

for i, entity in enumerate(sorted_entities[:20], 1):
    prob = f"{entity['min_prob']:.2f}" if entity['min_prob'] else "N/A"
    ent = f"{entity['max_entropy']:.2f}" if entity['max_entropy'] else "N/A"
    mar = f"{entity['min_margin']:.2f}" if entity['min_margin'] else "N/A"
    entity_text = entity['entity'][:33] + ".." if len(entity['entity']) > 35 else entity['entity']
    print(f"{i:<3} {entity_text:<35} {entity['type']:<18} {entity['risk_level']:<10} {prob:<7} {ent:<7} {mar:<7}")

if len(sorted_entities) > 20:
    print(f"... и ещё {len(sorted_entities) - 20} сущностей")

print("=" * 70)

# ============================================
# ЧАСТЬ 8: РЕКОМЕНДАЦИИ
# ============================================

print("\n💡 РЕКОМЕНДАЦИИ:")
print("-" * 70)

if report['negative_margin_count'] > 0:
    print(f"❗ {report['negative_margin_count']} сущностей с отрицательным Margin")
    print("   → Модель выбрала случайный токен. Требуется проверка врачом!")

if high_risk_percentage > 15:
    print(f"❗ {high_risk_percentage:.1f}% высокого риска (норма <15%)")
    print("   → Возможно, стоит улучшить промпт или проверить исходные данные")

if report['critical_high_risk_count'] > 0:
    print(f"❗ {report['critical_high_risk_count']} критических сущностей высокого риска")
    print("   → Рекомендуется ручная валидация перед использованием в клинике")

if high_risk_percentage <= 15 and report['negative_margin_count'] == 0:
    print("✅ Качество суммаризации в пределах нормы")

print("=" * 70)

# ============================================
# ЧАСТЬ 9: ЭКСПОРТ CSV ДЛЯ ВРАЧА (опционально)
# ============================================

import csv

csv_path = f"/content/critical_entities_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

priority_entities = report['negative_margin_entities'] + report['critical_high_risk']

if priority_entities:
    with open(csv_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Сущность', 'Тип', 'Риск', 'Prob', 'Entropy', 'Margin', 'Проверка'])

        for entity in priority_entities:
            writer.writerow([
                entity['entity'],
                entity['type'],
                entity['risk_level'],
                f"{entity['min_prob']:.3f}" if entity['min_prob'] else "N/A",
                f"{entity['max_entropy']:.3f}" if entity['max_entropy'] else "N/A",
                f"{entity['min_margin']:.3f}" if entity['min_margin'] else "N/A",
                "✅ ОБЯЗАТЕЛЬНО" if entity['min_margin'] is not None and entity['min_margin'] < 0 else "⚠️ Желательно"
            ])

    print(f"\n📄 CSV для врача сохранён: {csv_path}")

🔍 Анализ неопределённости модели...

📊 ОТЧЁТ ПО МЕТРИКАМ НЕОПРЕДЕЛЁННОСТИ
Всего сущностей: 109
🟢 Низкий риск: 30
🟡 Средний риск: 30
🔴 Высокий риск: 49
⚠️  Отрицательный Margin: 10
🏥 Определённые специальности: cardiology, neurology, gastroenterology

🚨 ТРЕБУЮТ СРОЧНОЙ ПРОВЕРКИ (отрицательный Margin):
----------------------------------------------------------------------
  • обострение хронического панкреатита (диагноз)
    Prob: 0.229 | Entropy: 0.499 | Margin: -0.030
  • Боль (лабораторный_показатель)
    Prob: 0.230 | Entropy: 0.465 | Margin: -0.105
  • Предварительный диагноз (лабораторный_показатель)
    Prob: 0.376 | Entropy: 0.326 | Margin: -0.244
  • Двусторонняя нижнедолевая (лабораторный_показатель)
    Prob: 0.244 | Entropy: 0.602 | Margin: -0.194
  • пачка (лабораторный_показатель)
    Prob: 0.462 | Entropy: 0.232 | Margin: -0.061
  • Холестерин общий (лабораторный_показатель)
    Prob: 0.203 | Entropy: 0.586 | Margin: -0.058
  • обострение хронического (лабораторный_показат

In [42]:
# @title Метрики качества суммаризации (АГРЕГАЦИЯ ПО СЛОВАМ)

import math
import json
import csv
import re
from typing import Dict, List, Optional, Set, Tuple
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# ============================================
# ЧАСТЬ 1: ФИЛЬТРЫ ТОКЕНОВ
# ============================================

class TokenFilter:
    """Фильтрация несмысловых токенов"""

    # Пунктуация и спецсимволы
    PUNCTUATION_PATTERN = re.compile(r'^[\s\.\,\;\:\!\?\(\)\[\]\{\}\"\'\-\–—\/\\@#\$\%\&\*\+\=\|\~\`\^\<\>\_\u2000-\u206F]+$')

    # Служебные токены JSON
    JSON_TOKENS = {'{', '}', '[', ']', ':', ',', '"'}

    # Предлоги и союзы (русский)
    STOP_WORDS = {
        'в', 'на', 'с', 'со', 'к', 'по', 'под', 'над', 'перед', 'при', 'через',
        'и', 'или', 'а', 'но', 'да', 'же', 'ли', 'бы', 'что', 'как', 'где', 'когда',
        'этот', 'эта', 'это', 'эти', 'тот', 'та', 'то', 'те', 'весь', 'вся', 'всё', 'все',
        'сам', 'сама', 'само', 'сами', 'себя', 'себе', 'собой',
        'для', 'без', 'из', 'от', 'до', 'за', 'про', 'о', 'об', 'обо',
        'ежедневно', 'предварительный', 'пред', 'ный', 'ист', 'ка', 'очки'
    }

    # Медицинские маркеры (приоритет)
    MEDICAL_MARKERS = {
        'панкреат', 'холецист', 'гастрит', 'язв', 'гепат', 'цирроз', 'кол',
        'карди', 'аритм', 'инфаркт', 'инсульт', 'онко', 'опухол', 'рак',
        'боль', 'отек', 'некроз', 'дилат', 'стенк', 'проток', 'пузырь',
        'печен', 'почк', 'желуд', 'кишечн', 'легк', 'сердц', 'сосуд'
    }

    @classmethod
    def is_meaningful(cls, token: str) -> bool:
        if not token or not token.strip():
            return False

        if cls.PUNCTUATION_PATTERN.match(token.strip()):
            return False

        if token.strip() in cls.JSON_TOKENS:
            return False

        if len(token.strip()) == 1 and not token.strip().isalpha():
            return False

        if token.strip().lower() in cls.STOP_WORDS:
            return False

        # Короткие токены (< 3 символов) — только если это начало медицинского термина
        if len(token.strip()) < 3:
            # Проверяем, не является ли токеном-префиксом медицинского термина
            token_lower = token.strip().lower()
            if not any(marker.startswith(token_lower) for marker in cls.MEDICAL_MARKERS):
                return False

        return True

    @classmethod
    def is_medical_priority(cls, token: str) -> bool:
        token_lower = token.strip().lower()
        return any(marker in token_lower for marker in cls.MEDICAL_MARKERS)


# ============================================
# ЧАСТЬ 2: АГРЕГАЦИЯ ТОКЕНОВ В СЛОВА
# ============================================

class WordAggregator:
    """Объединяет токены в слова для отчёта"""

    def __init__(self, tokens: List, context_window: int = 5):
        self.tokens = tokens
        self.context_window = context_window

    def _get_token_text(self, idx: int) -> str:
        return getattr(self.tokens[idx], 'token', '')

    def _logprob_to_prob(self, logprob: float) -> float:
        if logprob is None:
            return 0.0
        try:
            return math.exp(logprob)
        except (OverflowError, ValueError):
            return 0.0

    def _calculate_entropy(self, top_logprobs: List) -> float:
        if not top_logprobs:
            return 0.0

        probs = []
        for item in top_logprobs:
            logprob = getattr(item, 'logprob', None)
            if logprob is not None:
                prob = self._logprob_to_prob(logprob)
                if prob > 0:
                    probs.append(prob)

        if not probs:
            return 0.0

        total = sum(probs)
        if total > 0:
            probs = [p / total for p in probs]

        entropy = -sum(p * math.log(p) for p in probs if p > 0)
        max_entropy = math.log2(len(probs)) if len(probs) > 1 else 1.0
        normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0.0

        return min(normalized_entropy, 1.0)

    def _calculate_margin(self, top_logprobs: List) -> float:
        if len(top_logprobs) < 2:
            return 1.0

        p1 = self._logprob_to_prob(getattr(top_logprobs[0], 'logprob', None))
        p2 = self._logprob_to_prob(getattr(top_logprobs[1], 'logprob', None))

        return p1 - p2

    def _assess_risk(self, prob: Optional[float], entropy: Optional[float],
                     margin: Optional[float]) -> str:
        if prob is None or entropy is None or margin is None:
            return 'UNKNOWN'

        if margin < 0:
            return '🔴 HIGH'

        if prob < 0.5 or margin < 0.1 or entropy > 0.7:
            return '🔴 HIGH'

        if prob < 0.7 or margin < 0.3 or entropy > 0.5:
            return '🟡 MEDIUM'

        return '🟢 LOW'

    def _get_context(self, idx: int) -> str:
        start = max(0, idx - self.context_window)
        end = min(len(self.tokens), idx + self.context_window + 1)

        context_tokens = []
        for i in range(start, end):
            token_text = self._get_token_text(i)
            if i == idx:
                context_tokens.append(f"【{token_text}】")
            else:
                context_tokens.append(token_text)

        return ''.join(context_tokens)

    def analyze_tokens(self) -> List[Dict]:
        """Анализирует все токены"""
        results = []

        for idx in range(len(self.tokens)):
            token_data = self.tokens[idx]
            logprob = getattr(token_data, 'logprob', None)
            top_logprobs = getattr(token_data, 'top_logprobs', [])
            token_text = self._get_token_text(idx)

            prob = self._logprob_to_prob(logprob) if logprob is not None else None
            entropy = self._calculate_entropy(top_logprobs) if top_logprobs else None
            margin = self._calculate_margin(top_logprobs) if top_logprobs else None

            results.append({
                'token': token_text,
                'position': idx,
                'prob': round(prob, 4) if prob else None,
                'entropy': round(entropy, 4) if entropy else None,
                'margin': round(margin, 4) if margin else None,
                'risk_level': self._assess_risk(prob, entropy, margin),
                'context': self._get_context(idx),
                'is_meaningful': TokenFilter.is_meaningful(token_text),
                'is_medical': TokenFilter.is_medical_priority(token_text)
            })

        return results

    def aggregate_to_words(self, token_metrics: List[Dict]) -> List[Dict]:
        """
        Объединяет соседние токены в слова.
        Соседние токены — это токены без пробела между ними.
        """
        words = []
        current_word_tokens = []

        for i, token in enumerate(token_metrics):
            if not token['is_meaningful']:
                # Завершаем текущее слово
                if current_word_tokens:
                    words.append(self._aggregate_word(current_word_tokens))
                    current_word_tokens = []
                continue

            # Проверяем, начинается ли токен с пробела
            starts_with_space = token['token'].startswith(' ')

            if starts_with_space and current_word_tokens:
                # Новое слово
                words.append(self._aggregate_word(current_word_tokens))
                current_word_tokens = [token]
            else:
                # Продолжение текущего слова
                current_word_tokens.append(token)

        # Завершаем последнее слово
        if current_word_tokens:
            words.append(self._aggregate_word(current_word_tokens))

        return words

    def _aggregate_word(self, tokens: List[Dict]) -> Dict:
        """Агрегирует метрики токенов в слово"""
        word_text = ''.join(t['token'] for t in tokens).strip()

        # Агрегация метрик
        probs = [t['prob'] for t in tokens if t['prob'] is not None]
        entropies = [t['entropy'] for t in tokens if t['entropy'] is not None]
        margins = [t['margin'] for t in tokens if t['margin'] is not None]

        min_prob = min(probs) if probs else None
        max_entropy = max(entropies) if entropies else None
        min_margin = min(margins) if margins else None

        # Риск слова — максимальный риск среди токенов
        risk_levels = [t['risk_level'] for t in tokens]
        if '🔴 HIGH' in risk_levels:
            risk_level = '🔴 HIGH'
        elif '🟡 MEDIUM' in risk_levels:
            risk_level = '🟡 MEDIUM'
        else:
            risk_level = '🟢 LOW'

        # Контекст — берём от первого токена
        context = tokens[0]['context']

        # Медицинский приоритет
        is_medical = any(t['is_medical'] for t in tokens)

        return {
            'word': word_text,
            'start_position': tokens[0]['position'],
            'end_position': tokens[-1]['position'],
            'token_count': len(tokens),
            'min_prob': round(min_prob, 4) if min_prob else None,
            'max_entropy': round(max_entropy, 4) if max_entropy else None,
            'min_margin': round(min_margin, 4) if min_margin else None,
            'risk_level': risk_level,
            'context': context,
            'is_medical': is_medical
        }


# ============================================
# ЧАСТЬ 3: ЗАПУСК АНАЛИЗА
# ============================================

print("🔍 Анализ неопределённости модели (агрегация по словам)...\n")
print("=" * 70)

try:
    aggregator = WordAggregator(summary_logprobs, context_window=5)
    token_metrics = aggregator.analyze_tokens()
    word_metrics = aggregator.aggregate_to_words(token_metrics)
except NameError:
    print("❌ Ошибка: переменные summary_logprobs и summary_text не найдены!")
    raise

# ============================================
# ЧАСТЬ 4: ФИЛЬТРАЦИЯ ДЛЯ ОТЧЁТА
# ============================================

# 🔴 Жёсткий фильтр для отчёта
def is_critical_word(word: Dict) -> bool:
    if word['risk_level'] != '🔴 HIGH':
        return False

    # Обязательно показываем отрицательный margin
    if word['min_margin'] is not None and word['min_margin'] < 0:
        return True

    # Медицинские термины с высоким риском
    if word['is_medical'] and word['min_prob'] is not None and word['min_prob'] < 0.4:
        return True

    # Остальные — только если margin < 0
    return False

critical_words = [w for w in word_metrics if is_critical_word(w)]
negative_margin_words = [w for w in word_metrics
                         if w['min_margin'] is not None and w['min_margin'] < 0]

# Статистика
total_tokens = len(token_metrics)
meaningful_tokens = len([t for t in token_metrics if t['is_meaningful']])
total_words = len(word_metrics)
high_risk_words = len([w for w in word_metrics if w['risk_level'] == '🔴 HIGH'])

# ============================================
# ЧАСТЬ 5: ВЫВОД РЕЗУЛЬТАТОВ
# ============================================

print("📊 ОТЧЁТ ПО МЕТРИКАМ НЕОПРЕДЕЛЁННОСТИ")
print("=" * 70)
print(f"Всего токенов: {total_tokens}")
print(f"📝 Смысловых токенов: {meaningful_tokens}")
print(f"📚 Слов (агрегировано): {total_words}")
print(f"🔴 Высокий риск (слова): {high_risk_words}")
print(f"⚠️  Отрицательный Margin (слова): {len(negative_margin_words)}")
print(f"🎯 Критических для проверки: {len(critical_words)}")
print()

# 🔴 ПРИОРИТЕТ 1: Отрицательный Margin
if negative_margin_words:
    print("🚨 ТРЕБУЮТ СРОЧНОЙ ПРОВЕРКИ (отрицательный Margin):")
    print("-" * 70)
    for word in negative_margin_words[:8]:  # Показываем только топ-8
        prob_val = f"{word['min_prob']:.3f}" if word['min_prob'] else "N/A"
        ent_val = f"{word['max_entropy']:.3f}" if word['max_entropy'] else "N/A"
        mar_val = f"{word['min_margin']:.3f}" if word['min_margin'] else "N/A"
        med_marker = "🏥" if word['is_medical'] else "  "
        print(f"{med_marker} • '{word['word']}' (позиции {word['start_position']}-{word['end_position']})")
        print(f"    Контекст: {word['context']}")
        print(f"    Prob: {prob_val} | Entropy: {ent_val} | Margin: {mar_val}")
        print()
    if len(negative_margin_words) > 8:
        print(f"  ... и ещё {len(negative_margin_words) - 8} слов")
    print()

# 🔴 ПРИОРИТЕТ 2: Медицинские термины высокого риска
medical_high_risk = [w for w in critical_words if w['is_medical'] and w not in negative_margin_words]
if medical_high_risk:
    print("⚠️  МЕДИЦИНСКИЕ ТЕРМИНЫ ВЫСОКОГО РИСКА:")
    print("-" * 70)
    for word in medical_high_risk[:8]:  # Показываем только топ-8
        prob_val = f"{word['min_prob']:.3f}" if word['min_prob'] else "N/A"
        ent_val = f"{word['max_entropy']:.3f}" if word['max_entropy'] else "N/A"
        mar_val = f"{word['min_margin']:.3f}" if word['min_margin'] else "N/A"
        print(f"🏥 • '{word['word']}' (позиции {word['start_position']}-{word['end_position']})")
        print(f"    Контекст: {word['context']}")
        print(f"    Prob: {prob_val} | Entropy: {ent_val} | Margin: {mar_val}")
        print()
    if len(medical_high_risk) > 8:
        print(f"  ... и ещё {len(medical_high_risk) - 8} слов")
    print()

# ============================================
# ЧАСТЬ 6: СОХРАНЕНИЕ В JSON
# ============================================

high_risk_percentage = round(high_risk_words / max(total_words, 1) * 100, 2)

summary_result['uncertainty_metrics'] = {
    'summary': {
        'total_tokens': total_tokens,
        'meaningful_tokens': meaningful_tokens,
        'total_words': total_words,
        'high_risk_count': high_risk_words,
        'negative_margin_count': len(negative_margin_words),
        'critical_count': len(critical_words),
        'high_risk_percentage': high_risk_percentage
    },
    'priority_review': {
        'negative_margin': negative_margin_words,
        'medical_high_risk': medical_high_risk,
        'all_critical': critical_words
    },
    'all_words': word_metrics
}

with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_result, f, ensure_ascii=False, indent=2)

print(f"✅ Метрики сохранены в {summary_path}")
print("=" * 70)

# ============================================
# ЧАСТЬ 7: ТАБЛИЦА ТОП-15
# ============================================

print("\n📋 ТОП-15 СЛОВ ПО РИСКУ:")
print("-" * 70)
print(f"{'#': <3} {'Слово': <20} {'Контекст': <40} {'Риск': <10} {'Mar': <7}")
print("-" * 70)

sorted_words = sorted(word_metrics,
                      key=lambda x: (x['min_margin'] if x['min_margin'] else 1,
                                    0 if x['risk_level'] == '🔴 HIGH' else 1,
                                    0 if x['is_medical'] else 1))

for i, word in enumerate(sorted_words[:15], 1):
    mar = f"{word['min_margin']:.2f}" if word['min_margin'] else "N/A"
    ctx = word['context'][:38] + "..." if len(word['context']) > 40 else word['context']
    ctx = ctx.replace('\n', ' ')
    med_marker = "🏥" if word['is_medical'] else "  "
    print(f"{i: <3} {med_marker} {word['word']: <18} {ctx: <40} {word['risk_level']: <10} {mar: <7}")

if len(sorted_words) > 15:
    print(f"... и ещё {len(sorted_words) - 15} слов")

print("=" * 70)

# ============================================
# ЧАСТЬ 8: РЕКОМЕНДАЦИИ
# ============================================

print("\n💡 РЕКОМЕНДАЦИИ:")
print("-" * 70)

if len(negative_margin_words) > 0:
    print(f"❗ {len(negative_margin_words)} слов с отрицательным Margin")
    print("   → Модель выбрала случайный токен. Требуется проверка!")

if len(critical_words) > 0:
    print(f"❗ {len(critical_words)} критических слов для проверки")
    print("   → Сфокусируйтесь на этих терминах")

medical_count = len([w for w in critical_words if w['is_medical']])
if medical_count > 0:
    print(f"❗ {medical_count} медицинских терминов высокого риска")
    print("   → Приоритетная проверка врачом")

if high_risk_percentage > 15:
    print(f"❗ {high_risk_percentage:.1f}% высокого риска (норма <15%)")
    print("   → Возможно, стоит улучшить промпт")

if len(critical_words) == 0:
    print("✅ Качество суммаризации в пределах нормы")

print("=" * 70)

# ============================================
# ЧАСТЬ 9: ЭКСПОРТ CSV
# ============================================

csv_path = f"/content/critical_words_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

if critical_words:
    with open(csv_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Слово', 'Позиции', 'Контекст', 'Медицинский', 'Риск', 'Prob', 'Entropy', 'Margin', 'Проверка'])

        for word in critical_words:
            ctx_clean = word['context'].replace('\n', ' ').replace('\r', ' ')
            writer.writerow([
                word['word'],
                f"{word['start_position']}-{word['end_position']}",
                ctx_clean,
                "✅ ДА" if word['is_medical'] else "Нет",
                word['risk_level'],
                f"{word['min_prob']:.3f}" if word['min_prob'] else "N/A",
                f"{word['max_entropy']:.3f}" if word['max_entropy'] else "N/A",
                f"{word['min_margin']:.3f}" if word['min_margin'] else "N/A",
                "✅ ОБЯЗАТЕЛЬНО" if word['min_margin'] is not None and word['min_margin'] < 0 else "⚠️ Желательно"
            ])

    print(f"\n📄 CSV сохранён: {csv_path}")

🔍 Анализ неопределённости модели (агрегация по словам)...

📊 ОТЧЁТ ПО МЕТРИКАМ НЕОПРЕДЕЛЁННОСТИ
Всего токенов: 1889
📝 Смысловых токенов: 520
📚 Слов (агрегировано): 355
🔴 Высокий риск (слова): 31
⚠️  Отрицательный Margin (слова): 3
🎯 Критических для проверки: 3

🚨 ТРЕБУЮТ СРОЧНОЙ ПРОВЕРКИ (отрицательный Margin):
----------------------------------------------------------------------
🏥 • 'Боль' (позиции 38-38)
    Контекст:  "жалобы": "【Боль】 в животе разлитая
    Prob: 0.230 | Entropy: 0.465 | Margin: -0.105

   • 'Х' (позиции 1318-1318)
    Контекст: показатель": "【Х】олестерин общий
    Prob: 0.203 | Entropy: 0.586 | Margin: -0.058

   • 'боли' (позиции 1710-1710)
    Контекст: ерстной кишки как причину【 боли】 в животе.",
 
    Prob: 0.158 | Entropy: 0.563 | Margin: -0.176


✅ Метрики сохранены в /content/text1_source_summary_v6.json

📋 ТОП-15 СЛОВ ПО РИСКУ:
----------------------------------------------------------------------
#   Слово                Контекст                          

In [ ]:
# @title Экспорт критических сущностей для врача

import csv
from datetime import datetime

# Сущности для ручной проверки
priority_entities = report['negative_margin_entities'] + report['critical_high_risk']

# Экспорт в CSV
csv_path = f"/content/critical_entities_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

with open(csv_path, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Сущность', 'Тип', 'Risk', 'Prob', 'Entropy', 'Margin', 'Требуется проверка'])

    for entity in priority_entities:
        writer.writerow([
            entity['entity'],
            entity['type'],
            entity['risk_level'],
            f"{entity['min_prob']:.3f}" if entity['min_prob'] else "N/A",
            f"{entity['max_entropy']:.3f}" if entity['max_entropy'] else "N/A",
            f"{entity['min_margin']:.3f}" if entity['min_margin'] else "N/A",
            "✅ ДА" if entity['min_margin'] is not None and entity['min_margin'] < 0 else "⚠️ Желательно"
        ])

print(f"✅ CSV для врача сохранён: {csv_path}")

✅ CSV для врача сохранён: /content/critical_entities_20260329_094104.csv


---
# **Метрички и сигналки**